# Multimodal No-DA (Source-only) — accuracy + DTW(vs samsung1)

Source(samsung1)로만 학습된 멀티모달 baseline `weights/mm_no_da_seed42_best_model.pth`
(`InterFusionClassifier`)를 불러와, Target 디바이스 데이터 **전체**에 대한 윈도우 정확도와
samsung1 IMU 와의 DTW 를 측정한다.

5채널 노트북(`eval_baseline_tcn_source_only.ipynb`)과 차이:
- 모델: 5채널 TCN → **멀티모달 intermediate fusion** (EMG/IMU 분리 인코더)
- 추론: `U.load_mm_model` + `U.evaluate_mm`
- **IMU 를 네이티브 100Hz(500샘플)로 소비** — 5채널은 1000Hz 업샘플로 봤음

로직은 전부 `eval_utils.py` 에 있고, 이 노트북은 *DataFrame 을 로드해 넘기고 결과를 표로 정리*만 한다.
→ 같은 함수로 48가지 IMU 축 순열·부호 탐색까지 재사용 가능.

- `data/samsung2_YXZ.parquet`          : IMU 축이 samsung1 기준으로 정렬(YXZ 재라벨링)된 버전
- `data/samsung2.parquet` : `triceps_X` ↔ `triceps_Y` 를 되돌린 원본(축 미정렬, 주력)

> DTW 는 모델과 무관하게 raw IMU 컬럼으로 계산되므로 5채널 노트북과 같은 값이 나온다(축 정렬도 측정용).

In [1]:
import torch
DEV = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
import os, sys
import pandas as pd

ROOT = os.path.abspath('../..')
sys.path.insert(0, ROOT)
import eval_utils as U

# Source-only 멀티모달 모델 + samsung1 DTW reference (48회 재사용하므로 1회만 빌드)
MM_WEIGHTS = os.path.join(ROOT, 'weights', 'mm_no_da_seed42_best_model.pth')
model = U.load_mm_model(MM_WEIGHTS, device=DEV)
df_src = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung1.parquet'))
dtw_ref = U.build_dtw_reference(df_src)
print('MM model + reference ready | device:', next(model.parameters()).device)

MM model + reference ready | device: cuda:0


In [2]:
# 평가할 Target parquet 들 — 로드해서 DataFrame 으로 넘긴다
TARGETS = {
    'samsung2 (aligned)': 'samsung2_YXZ.parquet',
    'samsung2_original':  'samsung2.parquet',
}

results = {}
for name, fname in TARGETS.items():
    df_tgt = pd.read_parquet(os.path.join(ROOT, 'data', fname))
    results[name] = U.evaluate_mm(df_tgt, model, dtw_ref)   # {accuracy, dtw, n_windows}
    print(f'[{name}] {results[name]}')

[samsung2 (aligned)] {'accuracy': 59.031855379351796, 'dtw': 355.2044584326196, 'n_windows': 21629}
[samsung2_original] {'accuracy': 20.48638402145268, 'dtw': 364.5312512409307, 'n_windows': 21629}


In [3]:
# 요약
print('=' * 64)
print(f'{"dataset":<22}{"#windows":>10}{"accuracy":>12}{"DTW(vs s1)":>16}')
print('-' * 64)
for name, r in results.items():
    print(f'{name:<22}{r["n_windows"]:>10}{r["accuracy"]:>11.2f}%{r["dtw"]:>16.2f}')
print('=' * 64)

dataset                 #windows    accuracy      DTW(vs s1)
----------------------------------------------------------------
samsung2 (aligned)         21629      59.03%          355.20
samsung2_original          21629      20.49%          364.53
